In [1]:
import pandas as pd
import numpy as np
import random
from itertools import product

# 设置随机种子以确保可重复性
random.seed(42)
np.random.seed(42)

def create_negative_samples(positive_df, gene_list=None, allow_self_loop=False):
    """
    为基因调控网络构建负样本
    
    参数:
    positive_df: pandas DataFrame, 正样本数据，至少包含两列表示基因对
    gene_list: list, 所有基因的列表（可选）。如果为None，则从正样本中提取所有基因
    allow_self_loop: bool, 是否允许自环（基因对自身）。默认为False
    
    返回:
    negative_df: pandas DataFrame, 负样本数据
    """
    
    # 确保正样本数据有两列
    if len(positive_df.columns) < 2:
        raise ValueError("正样本DataFrame至少需要两列来表示基因对")
    
    # 获取基因对列名
    gene_col1, gene_col2 = positive_df.columns[:2]
    
    # 如果没有提供基因列表，从正样本中提取所有唯一基因
    if gene_list is None:
        tf_list = list(set(positive_df[gene_col1].tolist()))
        gene_list = list(set((positive_df[gene_col2].tolist())))

    # 创建所有可能的基因对（包括自环或不包括，根据参数）
    if allow_self_loop:
        all_possible_pairs = list(product(gene_list, repeat=2))
    else:
        all_possible_pairs = [(g1, g2) for g1 in tf_list for g2 in gene_list if g1 != g2]
    
    # 将正样本转换为集合以便快速查找
    positive_pairs = set(positive_df.apply(lambda row: tuple(row[:2]), axis=1))
    
    # 获取所有非正样本的基因对
    negative_candidates = [pair for pair in all_possible_pairs if pair not in positive_pairs]
    
    # 检查是否有足够的负样本候选
    n_positive = len(positive_df)
    if len(negative_candidates) < n_positive:
        print(f"警告: 负样本候选数量({len(negative_candidates)})少于正样本数量({n_positive})")
        print("将使用所有可用的负样本候选")
        n_samples = min(n_positive, len(negative_candidates))
    else:
        n_samples = n_positive
    
    # 随机抽取负样本
    selected_negative_pairs = random.sample(negative_candidates, n_samples)
    
    # 创建负样本DataFrame
    negative_df = pd.DataFrame(selected_negative_pairs, columns=[gene_col1, gene_col2])
    
    # 如果正样本有其他列，可以在负样本中添加对应的占位符列
    if len(positive_df.columns) > 2:
        for col in positive_df.columns[2:]:
            # 对于数值列，可以填充0；对于分类列，可以填充空值或特定值
            if pd.api.types.is_numeric_dtype(positive_df[col]):
                negative_df[col] = 0
            else:
                negative_df[col] = np.nan
    
    return negative_df

# 示例用法
if __name__ == "__main__":   
    
    pos = pd.read_csv("BL--network.csv")
    del pos["Score"]
    pos["label"] = 1
    pos.columns=["TF","Target","label"]
    # 方法1: 使用从正样本中提取的基因列表
    negative_df1 = create_negative_samples(pos)
    negative_df1["label"]=0
    negative_df1.columns=["TF","Target","label"]
    print(len(set(list(negative_df1["TF"]))))
    print(len(set(list(pos["TF"]))))
    print((set(list(pos["TF"]))))
    all=pd.concat([pos,negative_df1])   
    print(all)
    all.to_csv("AllPair.csv",index=False)

20
20
{'EGR1', 'JUNB', 'RELA', 'AHR', 'ETS2', 'ATF3', 'IRF4', 'HIF1A', 'MAFF', 'BATF3', 'IRF2', 'SUMO2', 'IRF1', 'NFKB1', 'RUNX1', 'STAT3', 'ATF4', 'STAT1', 'CTCF', 'SUMO1'}
        TF     Target  label
0      AHR  HIST1H2BC      1
1      AHR        CIC      1
2      AHR        SP1      1
3      AHR      ATP5B      1
4     ATF3      PSMB2      1
..     ...        ...    ...
751   ATF3     PTP4A2      0
752   ATF3       DPF2      0
753   ETS2      SIN3B      0
754   MAFF      CNOT6      0
755  STAT3     GLIPR1      0

[1512 rows x 3 columns]


In [1]:
import pandas as pd
a=pd.read_csv("ExpressionData.csv")
a

,CNOT6L,OST4,XBP1,DYNLT1B,MBD2,SMARCA5,H2-K1,HNRNPAB,PYCARD,PCNA,...,SIDT2,CIITA,RPL17,RPL21,PSMB9,ALOX5AP,RPL15,RPL37,AIF1,TCEB3
0,0.028261,3.691714,0.000000,0.000000,1.056347,0.129138,4.320748,2.224814,2.816441,1.591184,...,2.271733,0.008749,3.324659,3.041529,1.631825,3.649206,2.576738,3.111160,0.000000,0.604338
1,0.000000,3.516806,0.983792,0.000000,0.000000,0.945951,4.550070,1.591820,2.371083,2.728807,...,0.258387,0.231715,4.932846,4.364135,0.321921,3.085530,4.063016,4.257154,0.000000,0.376301
2,0.000000,3.853365,0.030061,0.000000,0.007357,0.046546,4.366995,1.353082,2.653665,3.313084,...,3.461425,0.000000,3.488372,3.131730,2.677448,4.123222,2.741763,3.090640,0.000000,0.000000
3,0.000000,3.605028,3.212447,1.212318,0.000000,1.099907,4.657145,1.078881,1.666923,1.347853,...,3.273905,0.000000,3.636250,2.813761,0.929459,3.956140,3.417909,2.853403,1.172886,0.006576
4,0.000000,4.342709,1.407312,0.000000,1.072724,1.972220,5.186013,1.811821,3.983589,4.324479,...,2.147830,0.150331,2.988212,2.848084,0.764208,5.416169,4.348650,3.100042,0.028622,0.525583
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
378,0.000000,2.645789,0.174163,2.882155,1.363470,1.124826,3.741048,0.822934,1.777015,3.189242,...,0.000000,0.000000,4.010471,2.615332,3.604510,2.651306,3.564188,3.241994,0.231120,2.358275
379,0.569255,1.738731,0.513217,1.797934,2.435603,0.750063,5.463508,0.000000,1.631026,0.991497,...,1.928483,0.000000,3.986062,3.274402,4.368818,3.225818,3.512854,3.479522,2.815370,0.248890
380,0.023096,4.078919,2.292473,0.000000,0.000000,0.598540,3.901505,0.894071,0.000000,1.191013,...,0.000000,1.529601,5.086163,3.179089,3.209996,4.019377,4.531868,3.387563,0.000000,0.246264
381,0.000000,4.063682,0.000000,0.000000,1.880526,0.386824,4.146143,0.826027,1.020479,2.522464,...,1.570086,1.478601,3.242416,2.441856,2.766577,2.013831,2.998692,2.956862,3.539482,0.081118
